# IA3 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [5]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [6]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [7]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [8]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [9]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [10]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")

In [11]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

"Human: 考试有哪些技巧？\n\nAssistant: 考试有哪些技巧？\n[Chastened Laugh]\nI told you it's not fun to be a guy.\n[Chastened Laugh]\nWhat it did for me was 触动了我.\nOh!\n[Chastened Laugh]\nThe more I saw of it, the more I was compelling.\n[Chastened Laugh]\nSo I got into it, got on the band.\nAnd when I look back, It's just the way I was.\nHow I was who I was.\nSo it's been a long time since I did"

## IA3

### PEFT Step1 配置文件

In [12]:
from peft import IA3Config, TaskType, get_peft_model

config = IA3Config(task_type=TaskType.CAUSAL_LM)
config

IA3Config(peft_type=<PeftType.IA3: 'IA3'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, target_modules=None, feedforward_modules=None, fan_in_fan_out=False, modules_to_save=None, init_ia3_weights=True)

### PEFT Step2 创建模型

In [13]:
model = get_peft_model(model, config)

In [14]:
config

IA3Config(peft_type=<PeftType.IA3: 'IA3'>, auto_mapping=None, base_model_name_or_path='../../model/langboat/bloom-1b4-zh', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, target_modules=['query_key_value', 'mlp.dense_4h_to_h'], feedforward_modules=['mlp.dense_4h_to_h'], fan_in_fan_out=False, modules_to_save=None, init_ia3_weights=True)

In [15]:
model

PeftModelForCausalLM(
  (base_model): IA3Model(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear(
                (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
                (ia3_l): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 6144x1 (cuda:0)])
              )
              (dense): Linear(in_features=2048, out_features=2048, bias=True)
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (mlp): BloomMLP(
       

In [16]:
model.print_trainable_parameters()

trainable params: 344,064 || all params: 1,303,455,744 || trainable%: 0.0264


## Step5 配置训练参数

In [17]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1,
    learning_rate=3e-3
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step6 创建训练器

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [19]:
trainer.train()

  0%|          | 0/3357 [00:00<?, ?it/s]

{'loss': 2.7946, 'grad_norm': 0.26694560050964355, 'learning_rate': 0.0029910634495084894, 'epoch': 0.0}
{'loss': 2.7807, 'grad_norm': 0.31290584802627563, 'learning_rate': 0.0029821268990169797, 'epoch': 0.01}
{'loss': 2.6015, 'grad_norm': 0.15633153915405273, 'learning_rate': 0.002973190348525469, 'epoch': 0.01}
{'loss': 2.5016, 'grad_norm': 0.2680078446865082, 'learning_rate': 0.002964253798033959, 'epoch': 0.01}
{'loss': 2.5317, 'grad_norm': 0.23941250145435333, 'learning_rate': 0.0029553172475424486, 'epoch': 0.01}
{'loss': 2.5504, 'grad_norm': 0.1743243783712387, 'learning_rate': 0.0029463806970509384, 'epoch': 0.02}
{'loss': 2.381, 'grad_norm': 0.1919475942850113, 'learning_rate': 0.002937444146559428, 'epoch': 0.02}
{'loss': 2.4374, 'grad_norm': 0.16738039255142212, 'learning_rate': 0.002928507596067918, 'epoch': 0.02}
{'loss': 2.2903, 'grad_norm': 0.23931531608104706, 'learning_rate': 0.0029195710455764074, 'epoch': 0.03}
{'loss': 2.2901, 'grad_norm': 0.1660180538892746, 'lear

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.1923, 'grad_norm': 0.20446689426898956, 'learning_rate': 0.002544235924932976, 'epoch': 0.15}
{'loss': 2.4257, 'grad_norm': 0.15678931772708893, 'learning_rate': 0.0025352993744414657, 'epoch': 0.15}
{'loss': 2.2092, 'grad_norm': 0.218997061252594, 'learning_rate': 0.0025263628239499555, 'epoch': 0.16}
{'loss': 2.2475, 'grad_norm': 0.2167639434337616, 'learning_rate': 0.0025174262734584452, 'epoch': 0.16}
{'loss': 2.2099, 'grad_norm': 0.1893223375082016, 'learning_rate': 0.0025084897229669346, 'epoch': 0.16}
{'loss': 2.155, 'grad_norm': 0.23844853043556213, 'learning_rate': 0.0024995531724754244, 'epoch': 0.17}
{'loss': 2.2601, 'grad_norm': 0.13333620131015778, 'learning_rate': 0.002490616621983914, 'epoch': 0.17}
{'loss': 2.3226, 'grad_norm': 0.2920139729976654, 'learning_rate': 0.002481680071492404, 'epoch': 0.17}
{'loss': 2.1706, 'grad_norm': 0.14512525498867035, 'learning_rate': 0.002472743521000894, 'epoch': 0.18}
{'loss': 2.1843, 'grad_norm': 0.22552219033241272, 'lear

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


...

{'loss': 2.0669, 'grad_norm': 0.31540197134017944, 'learning_rate': 0.0007569258266309205, 'epoch': 0.75}
{'loss': 2.09, 'grad_norm': 0.27159833908081055, 'learning_rate': 0.0007479892761394102, 'epoch': 0.75}
{'loss': 2.1904, 'grad_norm': 0.22204673290252686, 'learning_rate': 0.0007390527256479, 'epoch': 0.75}
{'loss': 2.2021, 'grad_norm': 0.18938203155994415, 'learning_rate': 0.0007301161751563897, 'epoch': 0.76}
{'loss': 2.0703, 'grad_norm': 0.1581420600414276, 'learning_rate': 0.0007211796246648794, 'epoch': 0.76}
{'loss': 2.0727, 'grad_norm': 0.14845731854438782, 'learning_rate': 0.000712243074173369, 'epoch': 0.76}
{'loss': 2.1321, 'grad_norm': 0.38207733631134033, 'learning_rate': 0.0007033065236818588, 'epoch': 0.77}
{'loss': 2.2515, 'grad_norm': 0.20283068716526031, 'learning_rate': 0.0006943699731903485, 'epoch': 0.77}
{'loss': 2.2737, 'grad_norm': 0.3125458061695099, 'learning_rate': 0.0006854334226988382, 'epoch': 0.77}
{'loss': 2.1759, 'grad_norm': 0.2327542006969452, 'lea

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.0982, 'grad_norm': 0.158440962433815, 'learning_rate': 0.0003100983020554066, 'epoch': 0.9}
{'loss': 2.1196, 'grad_norm': 0.25567135214805603, 'learning_rate': 0.00030116175156389633, 'epoch': 0.9}
{'loss': 2.1853, 'grad_norm': 0.26151472330093384, 'learning_rate': 0.0002922252010723861, 'epoch': 0.9}
{'loss': 2.0593, 'grad_norm': 0.17029568552970886, 'learning_rate': 0.00028328865058087577, 'epoch': 0.91}
{'loss': 2.1742, 'grad_norm': 0.22339944541454315, 'learning_rate': 0.0002743521000893655, 'epoch': 0.91}
{'loss': 2.1111, 'grad_norm': 0.15940417349338531, 'learning_rate': 0.00026541554959785526, 'epoch': 0.91}
{'loss': 2.2432, 'grad_norm': 0.13223670423030853, 'learning_rate': 0.00025647899910634495, 'epoch': 0.91}
{'loss': 2.1873, 'grad_norm': 0.15107962489128113, 'learning_rate': 0.0002475424486148347, 'epoch': 0.92}
{'loss': 2.2043, 'grad_norm': 0.19818006455898285, 'learning_rate': 0.0002386058981233244, 'epoch': 0.92}
{'loss': 2.0356, 'grad_norm': 0.177650347352027

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=3357, training_loss=2.1841030939015242, metrics={'train_runtime': 3154.2224, 'train_samples_per_second': 8.515, 'train_steps_per_second': 1.064, 'total_flos': 1.4768317912498176e+16, 'train_loss': 2.1841030939015242, 'epoch': 0.9999255342914588})

## Step8 模型推理

In [21]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧是指在考试的过程中如何提高效率、发挥优势或利用已知条件来提高成绩的技巧。大多数考试都包含多个部分，并且每个部分都存在一些特殊的方法和技巧，可以有助于提高成绩。例如，写作部分的技巧可以帮助考生更快地完成文章，而词汇量或写作技能的培训可以帮助考生更好地理解文章中的意思。因此，如果考生对这些技巧有深入的了解、运用得当，就能获得更好的成绩。'